# ==============================================================================
# PROYECTO INTEGRADOR PORTAFOLIO: COSTOS GESTIÓN.
# AUTOR : JORGE NIETO.
# TECNOLOGÍA: PYTHON + POSTGRESQL (NEON.TECH) + POWER BI.
# COMPONENTE: INGENIERÍA DE DATOS Y AUTOMATIZACIÓN DEL DATA WAREHOUSE
# ==============================================================================


In [7]:
# IMPORTACIÓN DE LIBRERÍAS ESTÁNDAR DE LA INDUSTRIA
import os  # Permite interactuar con el sistema operativo para leer variables ocultas (Secrets).
import pandas as pd  # La librería estándar para manipulación y estructura de datos en tablas (DataFrames).
import numpy as np  # Motor matemático utilizado aquí para simular la estacionalidad operativa.
from sqlalchemy import create_engine, text  # Motor para conectar a la base de datos cloud de Neon.

In [8]:
# ------------------------------------------------------------------------------
# CONEXIÓN AL MOTOR DE BASE DE DATOS SERVERLESS (NEON.TECH)
# ------------------------------------------------------------------------------
# Usamos la librería nativa de Colab para forzar la lectura del Secret de forma directa
from google.colab import userdata

try:
    DATABASE_URL = userdata.get('NEON_DB_URL')
except Exception:
    raise ValueError("⚠️ ERROR CRÍTICO: No se pudo leer la variable NEON_DB_URL. Verifica que el botón azul de los Secrets esté encendido.")

if not DATABASE_URL:
    raise ValueError("⚠️ ERROR CRÍTICO: La variable NEON_DB_URL está vacía.")

# Abrimos el canal de comunicación con los servidores de Neon
engine = create_engine(DATABASE_URL)

In [9]:
# ------------------------------------------------------------------------------
# CELDA 3: CREACIÓN DE ESTRUCTURAS DEL MODELO EN ESTRELLA (SQL NATIVO)
# ------------------------------------------------------------------------------
print("🚀 Iniciando reconstrucción idempotente del modelo en estrella en neondb...")

with engine.begin() as conn:
    # ELIMINACIÓN DE ESTRUCTURAS PREVIAS (Para evitar duplicaciones al reejecutar)
    conn.execute(text("DROP VIEW IF EXISTS public.costos1_vw_costeo_absorcion CASCADE;"))
    conn.execute(text("DROP VIEW IF EXISTS public.costos1_vw_costeo_variable CASCADE;"))
    conn.execute(text("DROP TABLE IF EXISTS public.costos1_fact_operaciones CASCADE;"))
    conn.execute(text("DROP TABLE IF EXISTS public.costos1_dim_tiempo CASCADE;"))
    conn.execute(text("DROP TABLE IF EXISTS public.costos1_dim_parametros CASCADE;"))

    # 1. CREACIÓN DE LA DIMENSIÓN TIEMPO
    conn.execute(text("""
        CREATE TABLE public.costos1_dim_tiempo (
            mes_id INT PRIMARY KEY,
            nombre_mes VARCHAR(20) NOT NULL,
            trimestre VARCHAR(5) NOT NULL
        );
    """))

    # 2. CREACIÓN DE LA DIMENSIÓN PARÁMETROS (Constantes fijas de la Cátedra)
    conn.execute(text("""
        CREATE TABLE public.costos1_dim_parametros (
            id_parametro SERIAL PRIMARY KEY,
            precio_unitario NUMERIC(10,2) NOT NULL,
            costo_variable_unitario NUMERIC(10,2) NOT NULL,
            cif_fijos_totales NUMERIC(12,2) NOT NULL,
            nap NUMERIC(10,2) NOT NULL,
            tasa_fija_cif NUMERIC(10,2) NOT NULL
        );
    """))

    # 3. CREACIÓN DE LA TABLA DE HECHOS (Operaciones mensuales de producción y venta)
    conn.execute(text("""
        CREATE TABLE public.costos1_fact_operaciones (
            id_operacion SERIAL PRIMARY KEY,
            mes_id INT REFERENCES public.costos1_dim_tiempo(mes_id),
            id_parametro INT REFERENCES public.costos1_dim_parametros(id_parametro),
            unidades_producidas INT NOT NULL,
            unidades_vendidas INT NOT NULL
        );
    """))

print("✅ Estructuras relacionales creadas con éxito en Neon.")

🚀 Iniciando reconstrucción idempotente del modelo en estrella en neondb...
✅ Estructuras relacionales creadas con éxito en Neon.


In [10]:
# ------------------------------------------------------------------------------
# CELDA 4: INYECCIÓN DE DATOS MAESTROS Y PARÁMETROS CONTABLES
# ------------------------------------------------------------------------------
print("📥 Insertando registros maestros en las dimensiones...")

# Cargamos la Dimensión Tiempo
meses_data = [
    (1, 'Enero', 'T1'), (2, 'Febrero', 'T1'), (3, 'Marzo', 'T1'),
    (4, 'Abril', 'T2'), (5, 'Mayo', 'T2'), (6, 'Junio', 'T2'),
    (7, 'Julio', 'T3'), (8, 'Agosto', 'T3'), (9, 'Septiembre', 'T3'),
    (10, 'Octubre', 'T4'), (11, 'Noviembre', 'T4'), (12, 'Diciembre', 'T4')
]
df_tiempo = pd.DataFrame(meses_data, columns=['mes_id', 'nombre_mes', 'trimestre'])
df_tiempo.to_sql('costos1_dim_tiempo', engine, if_exists='append', index=False)

# Cargamos los Parámetros del ejercicio (NAP = 10,000 unidades, Tasa Fija = $120)
parametros_data = {
    'precio_unitario': [1000.00],
    'costo_variable_unitario': [500.00],
    'cif_fijos_totales': [1200000.00],
    'nap': [10000.00],
    'tasa_fija_cif': [120.00]
}
df_parametros = pd.DataFrame(parametros_data)
df_parametros.to_sql('costos1_dim_parametros', engine, if_exists='append', index=False)

📥 Insertando registros maestros en las dimensiones...


1

In [11]:
# ------------------------------------------------------------------------------
# CELDA 5: SIMULADOR DE ESTACIONALIDAD OPERATIVA (PANDAS Y NUMPY)
# ------------------------------------------------------------------------------
print("⚙️ Ejecutando motor de simulación de demanda estacional...")
np.random.seed(42)  # Mantiene consistencia de datos históricos en el portafolio

# Simulación de la planta cordobesa de alfajores (Baja en verano, alta en invierno)
meses = np.arange(1, 13)
componente_estacional = np.sin((meses - 5) * (2 * np.pi / 12))

# Generación matemática de la Producción y Ventas reales por mes
produccion_real = (10500 + 2500 * componente_estacional + np.random.normal(0, 300, 12)).astype(int)
ventas_reales = (10200 + 2800 * componente_estacional + np.random.normal(0, 400, 12)).astype(int)

# Armamos el DataFrame final para la tabla de Hechos
operaciones_data = {
    'mes_id': list(range(1, 13)),
    'id_parametro': [1] * 12,  # Apunta al registro de parámetros contables fijos
    'unidades_producidas': produccion_real.tolist(),
    'unidades_vendidas': ventas_reales.tolist()
}
df_operaciones = pd.DataFrame(operaciones_data)
df_operaciones.to_sql('costos1_fact_operaciones', engine, if_exists='append', index=False)
print("✅ Tabla de hechos poblada con la simulación estacional.")

⚙️ Ejecutando motor de simulación de demanda estacional...
✅ Tabla de hechos poblada con la simulación estacional.


In [12]:
# ------------------------------------------------------------------------------
# CELDA 6: COMPILACIÓN DE MOTORES DE CÁLCULO CONTABLE (VISTAS SQL UNIFICADAS)
# ------------------------------------------------------------------------------
print("🔮 Compilando vistas de Costeo Variable y Absorción en Neon...")

with engine.begin() as conn:
    # VISTA 1: MODELO DE COSTEO VARIABLE (Margen de Contribución y Fijos al Período)
    conn.execute(text("""
        CREATE OR REPLACE VIEW public.costos1_vw_costeo_variable AS
        SELECT
            t.mes_id,
            t.nombre_mes,
            o.unidades_vendidas,
            (o.unidades_vendidas * p.precio_unitario) AS ingresos_ventas,
            (o.unidades_vendidas * p.costo_variable_unitario) AS costo_variable_total,
            ((o.unidades_vendidas * p.precio_unitario) - (o.unidades_vendidas * p.costo_variable_unitario)) AS margen_contribucion_total,
            p.cif_fijos_totales AS costos_fijos_periodo,
            (((o.unidades_vendidas * p.precio_unitario) - (o.unidades_vendidas * p.costo_variable_unitario)) - p.cif_fijos_totales) AS utilidad_variable
        FROM public.costos1_fact_operaciones o
        JOIN public.costos1_dim_tiempo t ON o.mes_id = t.mes_id
        JOIN public.costos1_dim_parametros p ON o.id_parametro = p.id_parametro;
    """))

    # VISTA 2: MODELO DE COSTEO POR ABSORCIÓN (Unificada, completa y sin cortes)
    conn.execute(text("""
        CREATE OR REPLACE VIEW public.costos1_vw_costeo_absorcion AS
        SELECT
            t.mes_id,
            t.nombre_mes,
            o.unidades_producidas,
            o.unidades_vendidas,
            (o.unidades_vendidas * p.precio_unitario) AS ingresos_ventas,
            (o.unidades_vendidas * (p.costo_variable_unitario + p.tasa_fija_cif)) AS costo_absorcion_vendido_estandar,
            ((o.unidades_producidas - p.nap) * p.tasa_fija_cif) AS variacion_capacidad,
            (o.unidades_vendidas * p.precio_unitario) - (o.unidades_vendidas * (p.costo_variable_unitario + p.tasa_fija_cif)) + ((o.unidades_producidas - p.nap) * p.tasa_fija_cif) AS utilidad_absorcion
        FROM public.costos1_fact_operaciones o
        JOIN public.costos1_dim_tiempo t ON o.mes_id = t.mes_id
        JOIN public.costos1_dim_parametros p ON o.id_parametro = p.id_parametro;
    """))

print("🌟 ¡PROCESO FINALIZADO EXITOSAMENTE! Tu Data Warehouse de Costos 1 está listo en Neon para conectarse a Power BI.")

🔮 Compilando vistas de Costeo Variable y Absorción en Neon...
🌟 ¡PROCESO FINALIZADO EXITOSAMENTE! Tu Data Warehouse de Costos 1 está listo en Neon para conectarse a Power BI.


In [15]:
# ==============================================================================
# CELDA 7: AUDITORÍA VISUAL Y EXPORTACIÓN DE ARCHIVOS PARA EL USUARIO
# ==============================================================================
print("🔍 Extrayendo datos desde Neon para auditoría local...")
engine.dispose() # Added to clear any pending transaction state

# 1. Extracción de las Vistas Contables compiladas en el servidor cloud
df_variable_cloud = pd.read_sql("SELECT * FROM public.costos1_vw_costeo_variable ORDER BY mes_id;", engine)
df_absorcion_cloud = pd.read_sql("SELECT * FROM public.costos1_vw_costeo_absorcion ORDER BY mes_id;", engine)

# 2. Visualización rápida en el cuaderno (Formato limpio)
print("\n📋 ESTADO DE RESULTADOS: MODELO DE COSTEO VARIABLE (Muestra)")
display(df_variable_cloud.head(3))

print("\n📋 ESTADO DE RESULTADOS: MODELO DE COSTEO POR ABSORCIÓN (Muestra)")
display(df_absorcion_cloud.head(3))

# 3. EXPORTACIÓN AUTOMÁTICA A FORMATOS DE NEGOCIO (CSV y Excel)
print("\n💾 Generando archivos de descarga local...")

# Exportación a archivos individuales CSV
df_variable_cloud.to_csv("Reporte_Costeo_Variable.csv", index=False, encoding="utf-8-sig")
df_absorcion_cloud.to_csv("Reporte_Costeo_Absorcion.csv", index=False, encoding="utf-8-sig")

# Exportación a un único libro de Excel con múltiples pestañas (Multisheet)
with pd.ExcelWriter("Auditoria_Costos1.xlsx", engine="openpyxl") as writer:
    df_variable_cloud.to_excel(writer, sheet_name="Costeo Variable", index=False)
    df_absorcion_cloud.to_excel(writer, sheet_name="Costeo por Absorcion", index=False)

print("✨ ¡Archivos creados exitosamente! Ya puedes descargarlos desde el panel izquierdo de Colab:")
print("   - Auditoria_Costos1_Batistella.xlsx (Recomendado para contadores)")
print("   - Reporte_Costeo_Variable.csv")
print("   - Reporte_Costeo_Absorcion.csv")

🔍 Extrayendo datos desde Neon para auditoría local...

📋 ESTADO DE RESULTADOS: MODELO DE COSTEO VARIABLE (Muestra)


,mes_id,nombre_mes,unidades_vendidas,ingresos_ventas,costo_variable_total,margen_contribucion_total,costos_fijos_periodo,utilidad_variable
0,1,Enero,7871,7871000.0,3935500.0,3935500.0,1200000.0,2735500.0
1,2,Febrero,6634,6634000.0,3317000.0,3317000.0,1200000.0,2117000.0
2,3,Marzo,7085,7085000.0,3542500.0,3542500.0,1200000.0,2342500.0



📋 ESTADO DE RESULTADOS: MODELO DE COSTEO POR ABSORCIÓN (Muestra)


,mes_id,nombre_mes,unidades_producidas,unidades_vendidas,ingresos_ventas,costo_absorcion_vendido_estandar,variacion_capacidad,utilidad_absorcion
0,1,Enero,8483,7871,7871000.0,4880020.0,-182040.0,2808940.0
1,2,Febrero,7958,6634,6634000.0,4113080.0,-245040.0,2275880.0
2,3,Marzo,8529,7085,7085000.0,4392700.0,-176520.0,2515780.0



💾 Generando archivos de descarga local...
✨ ¡Archivos creados exitosamente! Ya puedes descargarlos desde el panel izquierdo de Colab:
   - Auditoria_Costos1_Batistella.xlsx (Recomendado para contadores)
   - Reporte_Costeo_Variable.csv
   - Reporte_Costeo_Absorcion.csv
